# CS342 - Lab Assignment 1
## CUB-200-2011 Dataset — Class 009 (Brewer's Blackbird)

**Tasks:**
1. Download & explore dataset (class 009 only)
2. Segmentation algorithms + distance metrics
3. Edge detection — localization & false edges

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os, tarfile, urllib.request, shutil
from skimage import io, color, filters, morphology, segmentation, feature
from skimage.filters import sobel, roberts, prewitt, scharr, laplace, threshold_otsu
from skimage.segmentation import slic, felzenszwalb, watershed, mark_boundaries
from skimage.color import rgb2gray, label2rgb
from skimage.feature import canny
from skimage.metrics import variation_of_information, adapted_rand_error
from skimage.transform import resize
import warnings
warnings.filterwarnings('ignore')
print('Libraries loaded.')

---
## Step 0: Download CUB-200-2011 (Class 009 Only)

Downloads the full tar, extracts only `009.Brewer_Blackbird` images, then removes the tar to save space.  
**Run this cell once** — it skips if images already exist.

In [ ]:
DATA_DIR = 'cub009_images'
TAR_URL = 'https://data.caltech.edu/records/65de6-vp158/files/CUB_200_2011.tgz'
TAR_FILE = 'CUB_200_2011.tgz'
CLASS_PREFIX = 'CUB_200_2011/images/009.Brewer_Blackbird/'

if os.path.exists(DATA_DIR) and len(os.listdir(DATA_DIR)) > 0:
    print(f'Already have {len(os.listdir(DATA_DIR))} images in {DATA_DIR}/, skipping download.')
else:
    os.makedirs(DATA_DIR, exist_ok=True)
    
    # Download
    if not os.path.exists(TAR_FILE):
        print('Downloading CUB-200-2011 (~1.1 GB)... this takes a few minutes.')
        urllib.request.urlretrieve(TAR_URL, TAR_FILE)
        print('Download complete.')
    
    # Extract only class 009
    print('Extracting class 009 images...')
    with tarfile.open(TAR_FILE, 'r:gz') as tar:
        members = [m for m in tar.getmembers() if m.name.startswith(CLASS_PREFIX) and m.isfile()]
        for m in members:
            m.name = os.path.basename(m.name)  # flatten into DATA_DIR
            tar.extract(m, DATA_DIR)
    
    # Clean up tar
    os.remove(TAR_FILE)
    print(f'Done! Extracted {len(os.listdir(DATA_DIR))} images. Tar removed.')

---
## Task 1: Explore the Dataset

In [ ]:
# Load all images
image_files = sorted([f for f in os.listdir(DATA_DIR) if f.endswith('.jpg')])
all_images = [io.imread(os.path.join(DATA_DIR, f)) for f in image_files]

print(f'Class: 009.Brewer_Blackbird')
print(f'Total images: {len(all_images)}')
print(f'{"":-<60}')
print(f'{"Filename":<35s} {"Shape":>15s} {"Dtype":>8s}')
print(f'{"":-<60}')
for f, img in zip(image_files, all_images):
    print(f'{f:<35s} {str(img.shape):>15s} {str(img.dtype):>8s}')

In [ ]:
# Display all images in a grid
n = len(all_images)
cols = 5
rows = (n + cols - 1) // cols

fig, axes = plt.subplots(rows, cols, figsize=(16, rows * 3))
for i, ax in enumerate(axes.ravel()):
    if i < n:
        ax.imshow(all_images[i])
        ax.set_title(image_files[i][:20], fontsize=9)
    ax.axis('off')
plt.suptitle("CUB-200-2011 — Class 009: Brewer's Blackbird", fontsize=15, fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
# Image size distribution
heights = [img.shape[0] for img in all_images]
widths = [img.shape[1] for img in all_images]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(heights, bins=15, color='steelblue', edgecolor='black')
axes[0].set_title('Height Distribution'); axes[0].set_xlabel('Pixels')
axes[1].hist(widths, bins=15, color='coral', edgecolor='black')
axes[1].set_title('Width Distribution'); axes[1].set_xlabel('Pixels')
plt.suptitle('Image Size Statistics', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

print(f'Height — min: {min(heights)}, max: {max(heights)}, mean: {np.mean(heights):.0f}')
print(f'Width  — min: {min(widths)}, max: {max(widths)}, mean: {np.mean(widths):.0f}')

In [ ]:
# Color channel histograms for first image
sample = all_images[0]
fig, axes = plt.subplots(1, 4, figsize=(16, 3.5))
axes[0].imshow(sample); axes[0].set_title('Sample Image'); axes[0].axis('off')
for i, c in enumerate(['red', 'green', 'blue']):
    axes[i+1].hist(sample[:,:,i].ravel(), bins=64, color=c, alpha=0.7)
    axes[i+1].set_title(f'{c.capitalize()} Channel')
plt.suptitle('Color Distribution', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

---
## Task 2: Segmentation & Distance Metrics

We pick **one bird image** and apply 4 segmentation methods, then compute pairwise distance metrics.

In [ ]:
# Pick first image, resize to manageable size
img = all_images[0]
img_resized = (resize(img, (300, 400), anti_aliasing=True) * 255).astype(np.uint8)
gray = rgb2gray(img_resized)

# 1. SLIC
seg_slic = slic(img_resized, n_segments=100, compactness=10, start_label=1)

# 2. Felzenszwalb
seg_felz = felzenszwalb(img_resized, scale=100, sigma=0.5, min_size=50)

# 3. Watershed
gradient = sobel(gray)
markers = np.zeros_like(gray, dtype=int)
markers[gray < 0.3] = 1
markers[gray > 0.7] = 2
seg_water = watershed(gradient, markers)

# 4. Otsu
thresh = threshold_otsu(gray)
seg_otsu = (gray > thresh).astype(int) + 1

results = [('SLIC', seg_slic), ('Felzenszwalb', seg_felz),
           ('Watershed', seg_water), ('Otsu', seg_otsu)]

for name, seg in results:
    print(f'{name}: {len(np.unique(seg))} segments')

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
for i, (name, seg) in enumerate(results):
    axes[0, i].imshow(mark_boundaries(img_resized, seg, color=(1, 0, 0)))
    axes[0, i].set_title(f'{name} — Boundaries'); axes[0, i].axis('off')
    axes[1, i].imshow(label2rgb(seg, img_resized, kind='avg', bg_label=0))
    axes[1, i].set_title(f'{name} — Avg Color'); axes[1, i].axis('off')
plt.suptitle('Segmentation on Brewer\'s Blackbird', fontsize=15, fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
# Pairwise distance metrics
names = ['SLIC', 'Felzenszwalb', 'Watershed', 'Otsu']
segs = [seg_slic, seg_felz, seg_water, seg_otsu]
n = len(names)

vi_mat = np.zeros((n, n))
rand_mat = np.zeros((n, n))

for i in range(n):
    for j in range(n):
        if i != j:
            s, m = variation_of_information(segs[i], segs[j])
            vi_mat[i, j] = s + m
            are, _, _ = adapted_rand_error(segs[i], segs[j])
            rand_mat[i, j] = are

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, mat, title in zip(axes, [vi_mat, rand_mat],
    ['Variation of Information\n(lower = similar)', 'Adapted Rand Error\n(lower = similar)']):
    im = ax.imshow(mat, cmap='YlOrRd')
    ax.set_xticks(range(n)); ax.set_xticklabels(names, rotation=45)
    ax.set_yticks(range(n)); ax.set_yticklabels(names)
    ax.set_title(title, fontsize=12)
    for r in range(n):
        for c in range(n):
            ax.text(c, r, f'{mat[r,c]:.2f}', ha='center', va='center', fontsize=10)
    plt.colorbar(im, ax=ax, shrink=0.8)
plt.tight_layout(); plt.show()

In [ ]:
# Dice coefficient
def dice(s1, s2):
    b1 = (s1 > np.median(s1)).astype(bool)
    b2 = (s2 > np.median(s2)).astype(bool)
    return 2 * np.logical_and(b1, b2).sum() / (b1.sum() + b2.sum())

print('Dice Coefficient (higher = more similar):')
print(f'{"":>14s}', ''.join(f'{nm:>14s}' for nm in names))
for i in range(n):
    row = ''.join(f'{dice(segs[i], segs[j]):>14.4f}' for j in range(n))
    print(f'{names[i]:>14s}{row}')

---
## Task 3: Edge Detection
Apply 6 edge detectors, compare localization quality and false edge rates under noise.

In [ ]:
edges = {
    'Sobel': sobel(gray),
    'Prewitt': prewitt(gray),
    'Roberts': roberts(gray),
    'Scharr': scharr(gray),
    'Laplacian': np.abs(laplace(gray)),
    'Canny σ=1': canny(gray, sigma=1).astype(float),
    'Canny σ=2': canny(gray, sigma=2).astype(float),
    'Canny σ=3': canny(gray, sigma=3).astype(float),
}

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
for ax, (name, e) in zip(axes.ravel(), edges.items()):
    ax.imshow(e, cmap='gray'); ax.set_title(name, fontsize=12); ax.axis('off')
plt.suptitle('Edge Detection — Brewer\'s Blackbird', fontsize=15, fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
# Edge pixel stats
th = 0.1
print(f'{"Detector":<15s} {"Edge Pixels":>12s} {"Density %":>10s}')
print('-' * 40)
for name, e in edges.items():
    binary = e > 0.5 if 'Canny' in name else e > th
    count = binary.sum()
    print(f'{name:<15s} {count:>12d} {100*count/binary.size:>9.2f}%')

In [ ]:
# Zoomed comparison on a bird region
h, w = gray.shape
r0, r1 = h//4, 3*h//4
c0, c1 = w//4, 3*w//4

fig, axes = plt.subplots(2, 5, figsize=(20, 8))
axes[0,0].imshow(gray[r0:r1, c0:c1], cmap='gray')
axes[0,0].set_title('Original (zoomed)'); axes[0,0].axis('off')

selected = list(edges.keys())
positions = [(0,1),(0,2),(0,3),(0,4),(1,0),(1,1),(1,2),(1,3)]
for (r,c), name in zip(positions, selected):
    axes[r,c].imshow(edges[name][r0:r1, c0:c1], cmap='gray')
    axes[r,c].set_title(name, fontsize=11); axes[r,c].axis('off')
axes[1,4].axis('off')
plt.suptitle('Zoomed Edge Comparison (Center Region)', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
# Noise robustness
np.random.seed(42)
noisy = np.clip(gray + 0.1 * np.random.randn(*gray.shape), 0, 1)

noisy_edges = {
    'Sobel': sobel(noisy),
    'Roberts': roberts(noisy),
    'Canny σ=1': canny(noisy, sigma=1).astype(float),
    'Canny σ=3': canny(noisy, sigma=3).astype(float),
}

fig, axes = plt.subplots(1, 5, figsize=(20, 4))
axes[0].imshow(noisy, cmap='gray'); axes[0].set_title('Noisy Input'); axes[0].axis('off')
for ax, (name, e) in zip(axes[1:], noisy_edges.items()):
    ax.imshow(e, cmap='gray'); ax.set_title(name); ax.axis('off')
plt.suptitle('Edge Detection on Noisy Bird Image', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
# False edge rate
print('False Edge Rate (noisy vs clean):')
print('-' * 55)
for name in noisy_edges:
    clean = edges[name] > 0.5 if 'Canny' in name else edges[name] > th
    noisy_b = noisy_edges[name] > 0.5 if 'Canny' in name else noisy_edges[name] > th
    clean_d = morphology.binary_dilation(clean, morphology.disk(2))
    false_e = np.logical_and(noisy_b, ~clean_d).sum()
    total = noisy_b.sum()
    rate = 100 * false_e / total if total > 0 else 0
    print(f'{name:<15s} | False: {false_e:>5d} | Total: {total:>5d} | Rate: {rate:.1f}%')

---
## Conclusions

### Task 1 — Dataset Exploration:
- Class 009 (Brewer's Blackbird) contains ~60 images of varying sizes.
- Images are real-world bird photos with complex backgrounds — challenging for segmentation.

### Task 2 — Segmentation:
- **SLIC/Felzenszwalb** produce fine superpixels that follow bird contours.
- **Otsu** separates bird from background when contrast is sufficient.
- **Watershed** needs well-placed markers; default intensity thresholds may not isolate the bird well.
- VI & Rand Error show SLIC and Felzenszwalb are most mutually similar.

### Task 3 — Edge Detection:
- **Canny (σ=2)** provides the best balance of edge localization and low false edges.
- **Roberts** is sharpest but most noise-sensitive (highest false edge rate).
- **Sobel/Scharr** are good middle-ground choices.
- Higher Canny σ suppresses noise but loses feather/texture detail on the bird.